In [29]:
import pandas as pd
import re
from datetime import datetime

In [22]:
subreddits = {"4chan": "4chan_09_11.jsonl",
              "Antiwork": "Antiwork_10_11.jsonl",
              "AsianMasculinity": "AsianMasculinity_11_11.jsonl",
              "aznidentity": "aznidentity_11_11.jsonl",
              "BlackPeopleTwitter": "BlackPeopleTwitter_11_11.jsonl",
              "climateskeptics": "climateskeptics_12_11.jsonl",
              "conservatives": "conservatives_12_11.jsonl",
              "conspiracy": "conspiracy_12_11.jsonl",
              "GoldandBlack": "GoldandBlack_12_11.jsonl",
              "HermanCainAward": "HermanCainAward_12_11.jsonl",
              "KotakuInAction": "KotakuInAction_13_11.jsonl",
              "MensRights": "MensRights_13_11.jsonl",
              "NoFap": "NoFap_13_11.jsonl",
              "prolife": "prolife_13_11.jsonl",
              "TrueUnpopularOpinion": "TrueUnpopularOpinion_13_11.jsonl",
              "WhitePeopleTwitter": "WhitePeopleTwitter_14_11.jsonl"
             } # subreddits dictionnary with the subreddit names as its keys and the corresponding files collected through the API as its values

dataframes = []

for subreddit in subreddits:
    df = pd.read_json(subreddits[subreddit], lines=True) # turns each jsonl file into a DataFrame
    df["subreddit"] = subreddit # creates a subreddit column in the DataFrames to keep track of it
    dataframes.append(df) # creates a list of subreddit DataFrames

In [23]:
posts = pd.concat(dataframes) # concatenates the DataFrames list into one DataFrame

posts["created_utc"] = pd.to_datetime(posts["created_utc"], unit="s") # turns each post's UTC timestamp into a datetime object
too_old = posts["created_utc"] < datetime(2023, 1, 1) # localizes the posts from before 2023 (there were only 5)
posts = posts[~too_old] # removes the posts from before 2023

posts = posts.reset_index(drop=True) # resets the index column that was left in shambles after the concatenation and the deletion

In [24]:
posts.sample(10)

,id,title,selftext,created_utc,score,comments,subreddit
6352,1x20fce,Bill Gates flies on a private jet everywhere …,,2026-10-09 23:37:45,250,"[{'id': 'peytsv6', 'parent_id': 't3_1x20fce', ...",climateskeptics
2348,1i79k8z,"X, Meta, and CCP-affiliated content is no long...","Hello, everyone! Following recent events in so...",2025-01-22 12:02:29,49278,"[{'id': 'm8iwgc3', 'parent_id': 't3_1i79k8z', ...",Antiwork
21856,1vco46m,James Talarico speaks truth,,2026-08-01 13:57:20,20602,"[{'id': 'p12keeo', 'parent_id': 't3_1vco46m', ...",WhitePeopleTwitter
14107,1puonn3,ex-Bungie next to nu-Bungie,,2025-12-24 14:30:29,700,"[{'id': 'nvq8bna', 'parent_id': 't3_1puonn3', ...",KotakuInAction
13627,1w69620,Seems like Ayakamods is building their own Ste...,,2026-09-03 14:34:38,413,"[{'id': 'p7le49z', 'parent_id': 't3_1w69620', ...",KotakuInAction
11288,1uwkui0,How Americans governed themselves as British c...,,2026-07-14 20:49:18,8,[],GoldandBlack
4566,1s6nzu4,Saw some posts from Asian American feminists —...,Asian American feminists often claim Asian men...,2026-03-29 06:46:00,198,"[{'id': 'od595db', 'parent_id': 't3_1s6nzu4', ...",aznidentity
5437,1vy72ry,You can bet your ass the British tabloids are ...,,2026-08-25 17:53:00,904,"[{'id': 'p5uot33', 'parent_id': 't3_1vy72ry', ...",BlackPeopleTwitter
21882,1sjslwz,"1 down, 3 to go",,2026-04-12 22:23:55,20070,"[{'id': 'ofu20ie', 'parent_id': 't3_1sjslwz', ...",WhitePeopleTwitter
21946,1wwmnng,The Cornell 57…,,2026-10-03 12:59:26,18958,"[{'id': 'pdlqczw', 'parent_id': 't3_1wwmnng', ...",WhitePeopleTwitter


In [25]:
comments_dataframes = []

for x in posts.index:
    df = pd.DataFrame(posts.loc[x, "comments"]) # turns the "comments" list of dictionnaries of each post into a DataFrame
    df["subreddit"] = posts.loc[x, "subreddit"] # again, creates a "subreddit" column into each DataFrame
    comments_dataframes.append(df)

In [49]:
comments = pd.concat(comments_dataframes)

comments['created_utc'] = pd.to_datetime(comments['created_utc'], unit='s')

comments = comments.reset_index(drop=True)


#checks for deleted comments 
deleted_comments = comments[comments["body"] == "[deleted]"]
print(deleted_comments.head())
print(f"There's {len(deleted_comments)} deleted comments in the comments dataset.")
#print(len(comments))

#the comments variable is now without any deleted comments
comments = comments[comments["body"] !="[deleted]"]
print(len(comments))
#print(len(comments))


                id         parent_id       body         created_utc  score  \
344  19sbuv872d735  t1_19sbu638fo2sg  [deleted] 2026-11-08 13:55:17    1.0   
446  19sarxcmhanus  t1_19saonxqtkls3  [deleted] 2026-11-07 23:22:13   -3.0   
576  19s9acjpit091  t1_19s96ttiwrbsw  [deleted] 2026-11-07 03:20:59   -8.0   
653  19s9bnlsa89kx  t1_19s9b7crl31q8  [deleted] 2026-11-07 03:50:18   -6.0   
688  19safkkr0q5xc  t1_19sabq29guidc  [deleted] 2026-11-07 18:45:13    1.0   

    subreddit  
344     4chan  
446     4chan  
576     4chan  
653     4chan  
688     4chan  
There's 19128 deleted comments in the comments dataset.
1798575


In [ ]:

#regular expression to delete all the links and replace them by URL REMOVED 
comments['body'] = comments['body'].apply(lambda x: re.sub(r'(?:https?://|www\.)[^\s<>"\']+', "[URL REMOVED]", x))


#comments whitout the ones containing urls 
comments = comments[comments["body"] !="[URL REMOVED]"]
print(len(comments))



0
1788844


In [42]:
comments.sample(10)

,id,parent_id,body,created_utc,score,subreddit
1683486,ods66h0,t3_1s9y4m6,But if a fortune 500 needs a billion dollar ta...,2026-04-01 22:42:01,6.0,WhitePeopleTwitter
225188,oyr95xg,t1_oyp14wk,If only management would “choose to be respons...,2026-07-20 21:50:23,18.0,Antiwork
848777,nssktwc,t1_nssgv24,That's one of the things that gets me. They've...,2025-12-07 17:42:11,32.0,conspiracy
840642,owrsa3q,t1_owri0dh,We should make buildings out of those passport...,2026-07-10 20:36:55,64.0,conspiracy
970167,19rdspvwm5kwz,t1_19rdpnuzrpxy9,Say cheese!,2026-10-20 11:44:50,1.0,conspiracy
927380,oeyadrm,t1_oewgkhw,No no. They didn’t have to resort to blackmail...,2026-04-08 07:02:02,67.0,conspiracy
1615965,19s7h82tla7z4,t1_19s7f3crzy8rm,"lol, that’s it",2026-11-06 03:00:54,1.0,WhitePeopleTwitter
813908,o3eyv7i,t1_o3eouqw,probs nazis and japanese ww2 scientists. there...,2026-02-03 21:27:08,11.0,conspiracy
1339135,19sfyt7uwfvs5,t1_19sfyo092haqt,"Corruption is corruption, I'd rather corrupt p...",2026-11-10 21:12:13,1.0,TrueUnpopularOpinion
1408091,p5j5eh0,t1_p5j54k2,That’s when she was still in psychosis. Crazy...,2026-08-24 03:12:58,46.0,TrueUnpopularOpinion


In [28]:
#we verify if we have duplicates in the posts and comments

posts = posts.drop(columns=["comments"])
all_dups_p = posts[posts.duplicated(keep=False)] #keep False pour marquer toutes les occurences d'un doublon
print('Rows involved in duplicates in posts:', len(all_dups_p))

all_dups_c = comments[comments.duplicated(keep=False)] 
print('Rows involved in duplicates in comments:', len(all_dups_c))


Rows involved in duplicates in posts: 0
Rows involved in duplicates in comments: 0


In [ ]:
text = pd.concat([posts["title"], posts["selftext"], comments["body"]], ignore_index=True) # creates a DataFrame that only contains the text bearing columns

text.to_csv("data.csv", index=False, header=["text"], encoding="utf-8") # turns the text DataFrame into a csv file